# Alpha Engine — v6 corrected-price Colab research preparation

**Original v5 KLAC/CRWD prices were contaminated. Its 261 completed tasks are engineering evidence only.** This template now pins corrected v6 input hashes; use a new batch ID and fresh checkpoint parent. Do not restore the old checkpoint against v6. Public training opt-ins remain OFF. The Agent did not retrain or access Colab.

# Alpha Engine — Colab Multi-model Research Campaign

Prepare and manually execute **XGBoost, LightGBM and a fixed 20-day momentum rule**, reproducible bounded parameter search, feature ablations, seed/fold repetition and cost stress using one existing Study Workspace evaluator.

The example freezes **29 configurations / 261 fold-seed tasks**, including **24 sampled hyperparameter configurations**, three expanding validation folds and three seeds. July 1–August 4, 2026 is excluded from all search evaluators. Historical non-consumption of that reserved period is **not certified**.

Public template opt-ins are OFF. Owner launches this interactive notebook personally. No scheduler, automatic model promotion, US Alpha158 synthesis or claim of model efficacy. CPU first; choosing a GPU runtime alone does not enable GPU training or establish a speed benefit.


In [ ]:
from pathlib import Path
import hashlib, json, re, shutil, subprocess, sys

# Configure paths in your *private* Drive. No folder IDs, tokens, or sensitive
# metadata should be committed to GitHub.
V5_DIR = Path('/content/drive/MyDrive/CHANGE_ME/private/input_packages/us87-cn130-20261103-local-v6-us87-fullrefresh')
BATCH_SPEC_FILE = Path('/content/drive/MyDrive/CHANGE_ME/private/us87_model_search.json')
CLOUD_RIGHTS_RECORD = Path('/content/drive/MyDrive/CHANGE_ME/private/us87_cloud_rights.json')
CHECKPOINT_PARENT = Path('/content/drive/MyDrive/CHANGE_ME/private/incoming_unreviewed')

EXPECTED_MANIFEST_SHA = '4f8225f3c68ee0886b9b707ca140e91dcb509b739b8548fdc10f258574d3da1a'
EXPECTED_US_HISTORY_SHA = 'b4745e46fc26b0328e22de50e579ebbf1d400996258ea6ebb43c1ff739ed5a1a'
VERIFIER_AND_MODEL_RUNNER_GIT_SHA = 'CHANGE_ME_EXACT_MODEL_EXPLORATION_COMMIT_SHA'
EXPECTED_BATCH_SPEC_SHA256 = 'CHANGE_ME_64_CHARACTER_BATCH_FILE_SHA256'
EXPECTED_CLOUD_RIGHTS_SHA256 = 'CHANGE_ME_64_CHARACTER_RIGHTS_FILE_SHA256'
UV_VERSION = '0.9.10'
PYTHON_VERSION = '3.12.12'

OPERATOR_APPROVES_PRIVATE_CLOUD_DATA_AND_TRAINING = False
RUN_ONE_TRAINING_CHUNK = False
EXPORT_IMMUTABLE_CHECKPOINT = False
CHUNK_TRIAL_BUDGET = 6
CHUNK_WALL_SECONDS = 300

# Optional exact ZIP from a previous immutable checkpoint; never scan
# unattended or automatically select a "latest" checkpoint.
RESUME_FROM_VERIFIED_CHECKPOINT = None
EXPECTED_CHECKPOINT_INDEX_SHA256 = None
EXPECTED_CHECKPOINT_ARCHIVE_SHA256 = None

assert OPERATOR_APPROVES_PRIVATE_CLOUD_DATA_AND_TRAINING, 'Owner opt-in required'
assert all('CHANGE_ME' not in str(p) for p in
           (V5_DIR, BATCH_SPEC_FILE, CLOUD_RIGHTS_RECORD, CHECKPOINT_PARENT))
assert re.fullmatch(r'[a-f0-9]{40}', VERIFIER_AND_MODEL_RUNNER_GIT_SHA)
assert 1 <= CHUNK_TRIAL_BUDGET <= 20
assert re.fullmatch(r'[a-f0-9]{64}', EXPECTED_BATCH_SPEC_SHA256)
assert re.fullmatch(r'[a-f0-9]{64}', EXPECTED_CLOUD_RIGHTS_SHA256)
assert 1 <= CHUNK_WALL_SECONDS <= 7200

## 1 · Rights review and private Drive mount

A provider-specific authorization record must be reviewed by the owner. Permission to mount Drive is **not** permission from Yahoo or any other upstream price-data source. If the intended cloud computation is not licensed, stop before mounting/copying and use local-only execution.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

assert hashlib.sha256(CLOUD_RIGHTS_RECORD.read_bytes()).hexdigest() == EXPECTED_CLOUD_RIGHTS_SHA256
rights = json.loads(CLOUD_RIGHTS_RECORD.read_text(encoding='utf-8'))
assert rights.get('schema_version') == 'e1a_price_cloud_rights_v1'
assert rights.get('market') == 'us'
assert rights.get('bundle_manifest_sha256') == EXPECTED_MANIFEST_SHA
assert rights.get('history_archive_sha256') == EXPECTED_US_HISTORY_SHA
assert rights.get('cloud_processing_purpose') == 'private_colab_daily_price_research'
assert rights.get('owner_approved_private_cloud_processing') is True
assert rights.get('provider_rights') and all(
    p.get('private_cloud_compute_permitted') is True and
    str(p.get('terms_url', '')).startswith('https://') and
    len(str(p.get('review_basis', '')).strip()) >= 12
    for p in rights['provider_rights']
)
print('Owner-attested rights record present (not independent vendor certification).')
assert {p['provider'] for p in rights['provider_rights']} >= {'yfinance'}
print('Documentary proof path:', rights.get('documentary_proof_path') or 'Not supplied; owner attestation only.')

## 2 · Copy exactly the authorized price input once, then pin code and data

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

scratch = Path('/content/p2_e2_scratch')
scratch.mkdir(parents=True, exist_ok=False)
for src, name, expected in [
    (V5_DIR/'bundle-manifest.json', 'bundle-manifest.json', EXPECTED_MANIFEST_SHA),
    (V5_DIR/'us_market_history.zip', 'us_market_history.zip', EXPECTED_US_HISTORY_SHA)
]:
    dest = scratch/name
    shutil.copyfile(src, dest)
    assert sha256_file(dest) == expected, f'external byte pin mismatch: {name}'
batch_file = scratch/'batch.json'
shutil.copyfile(BATCH_SPEC_FILE, batch_file)
assert sha256_file(batch_file) == EXPECTED_BATCH_SPEC_SHA256
batch = json.loads(batch_file.read_text(encoding='utf-8'))
assert batch['runner_git_sha'] == VERIFIER_AND_MODEL_RUNNER_GIT_SHA
assert batch['v5_bundle_manifest_sha256'] == EXPECTED_MANIFEST_SHA
assert batch['history_archive_sha256'] == EXPECTED_US_HISTORY_SHA
assert batch['purpose'] == 'validation_search_only'

checkout = Path('/content/p2_e2_checkout')
subprocess.run(['git','clone','--quiet','--filter=blob:none','--no-checkout',
                'https://github.com/liuh886/alpha_engine.git',str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'sparse-checkout','init','--cone'],check=True)
subprocess.run(['git','-C',str(checkout),'sparse-checkout','set','src'],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',
                VERIFIER_AND_MODEL_RUNNER_GIT_SHA],check=True)
assert subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],
                               text=True).strip() == VERIFIER_AND_MODEL_RUNNER_GIT_SHA
subprocess.run([sys.executable,'-m','pip','install','--quiet',f'uv=={UV_VERSION}'],check=True)
subprocess.run(['uv','sync','--frozen','--python',PYTHON_VERSION],cwd=checkout,check=True)
# Model work runs inside the frozen uv project, not Colab's ambient kernel.
environment_probe = "import json,platform,sys,os,importlib.metadata as m;print(json.dumps({'python':platform.python_version(),'platform':platform.platform(),'cpu_count':os.cpu_count(),'packages':{p:m.version(p) for p in ['numpy','pandas','scipy','xgboost','lightgbm']}}))"
environment = json.loads(subprocess.check_output(
    ['uv','run','--frozen','python','-c',environment_probe],cwd=checkout,text=True))
assert environment['python'] == PYTHON_VERSION
print(json.dumps(environment,indent=2))


## 3 · Verify v5 bytes and build an isolated, non-Git research data root

The original dataset remains immutable and local-only. Cloud permission is a separate private operator record, never a mutation to v5.

In [ ]:
def uv_module(*args):
    result = subprocess.run(['uv','run','--frozen','python','-m',*map(str,args)],
                            cwd=checkout,capture_output=True,text=True)
    if result.returncode:
        print(result.stderr[-4000:])
        result.check_returncode()
    payload = json.loads(result.stdout)
    print(json.dumps({k:v for k,v in payload.items() if k not in ('trials','receipt','configurations')},indent=2))
    return payload

uv_module('src.research.e1a_price_preflight',
          '--manifest',scratch/'bundle-manifest.json',
          '--archive',scratch/'us_market_history.zip',
          '--expected-manifest-sha256',EXPECTED_MANIFEST_SHA,
          '--expected-archive-sha256',EXPECTED_US_HISTORY_SHA,
          '--rights-record',CLOUD_RIGHTS_RECORD,'--require-cloud-rights')
stage_root = scratch/'isolated_us87'
uv_module('src.research.v5_isolated_staging','stage',
          '--manifest',scratch/'bundle-manifest.json',
          '--archive',scratch/'us_market_history.zip',
          '--expected-manifest-sha256',EXPECTED_MANIFEST_SHA,
          '--expected-archive-sha256',EXPECTED_US_HISTORY_SHA,
          '--output-root',stage_root)
uv_module('src.research.colab_experiment_batch','plan',
          '--batch-spec',batch_file)
print('Verified complete US87 input; no model execution yet.')

## 4 · Optional: restore one explicitly selected checkpoint ZIP

Select a trusted private `incoming_unreviewed` ZIP and retain both its archive and inner-index SHA256 independently from the preceding export receipt. Archive size, safe members, code/data/batch identities and all receipt hashes are verified before restoring outputs. Never automatically select a latest checkpoint.

In [ ]:
output_root = scratch/'runs'
resume_flag = False
last_checkpoint = None
if RESUME_FROM_VERIFIED_CHECKPOINT is not None:
    assert re.fullmatch(r'[a-f0-9]{64}', str(EXPECTED_CHECKPOINT_INDEX_SHA256))
    assert re.fullmatch(r'[a-f0-9]{64}', str(EXPECTED_CHECKPOINT_ARCHIVE_SHA256))
    local_archive = scratch/'resume-checkpoint.zip'
    shutil.copyfile(Path(RESUME_FROM_VERIFIED_CHECKPOINT), local_archive)
    uv_module('src.research.colab_experiment_batch','restore-archive',
              '--batch-spec',batch_file,'--data-root',stage_root,
              '--archive',local_archive,
              '--expected-archive-sha256',EXPECTED_CHECKPOINT_ARCHIVE_SHA256,
              '--expected-index-sha256',EXPECTED_CHECKPOINT_INDEX_SHA256,
              '--output-root',output_root)
    resume_flag = True


def save_checkpoint():
    import datetime
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    name = 'e2-us87-checkpoint-'+stamp+'.zip'
    local_archive = scratch/name
    receipt = uv_module('src.research.colab_experiment_batch','checkpoint-archive',
                       '--batch-spec',batch_file,'--data-root',stage_root,
                       '--output-root',output_root,'--archive',local_archive)
    CHECKPOINT_PARENT.mkdir(parents=True, exist_ok=True)
    dest = CHECKPOINT_PARENT/name
    with local_archive.open('rb') as inp, dest.open('xb') as out:
        shutil.copyfileobj(inp, out)
    assert sha256_file(dest) == receipt['archive_sha256'], 'Drive readback SHA mismatch'
    assert dest.stat().st_size == receipt['archive_size_bytes']
    receipt['checkpoint'] = str(dest)
    receipt['drive_filesystem_readback_verified'] = True
    return receipt


## 5 · Run one bounded exploration chunk

Each invocation starts at most `CHUNK_TRIAL_BUDGET` new tasks and stops starting new work after `CHUNK_WALL_SECONDS`. A running native fit may finish after that start deadline; this is not a hard-kill timer. Three consecutive model errors stop new attempts and retain the failure receipts. Search tasks are interleaved so the first chunk covers the rule baseline, both native rankers and controlled feature comparisons.

With checkpoint export enabled, each completed chunk is saved immediately as private incoming_unreviewed evidence. Re-run this training cell to continue; never launch overlapping writers for the same output root.


In [ ]:
if RUN_ONE_TRAINING_CHUNK:
    args = ['src.research.colab_experiment_batch','run',
            '--batch-spec',batch_file,'--data-root',stage_root,
            '--output-root',output_root,'--max-new-trials',CHUNK_TRIAL_BUDGET,
            '--max-wall-seconds',CHUNK_WALL_SECONDS]
    if resume_flag:
        args.append('--resume')
    result = uv_module(*args)
    resume_flag = True
    if EXPORT_IMMUTABLE_CHECKPOINT:
        last_checkpoint = save_checkpoint()
        print('Save this checkpoint ZIP path, archive SHA256 and independent index SHA256 before resetting the runtime.')
else:
    print('No model fit: RUN_ONE_TRAINING_CHUNK is False.')


## 6 · Inspect the private, unreviewed checkpoint

Each completed chunk seals the existing strict receipt inventory locally and transfers **one ZIP** to Drive, then checks its size and SHA256 by reading it back. This avoids hundreds of small Drive writes. Checkpoints retain a 20 MiB uncompressed results budget; raw source data, extra files, changed identities and unsafe ZIP members are refused.

Record the returned ZIP path, `archive_sha256` and `index_sha256` separately. In a fresh runtime, set all three restore variables in the first cell. Completed trials are skipped; an atomic study interrupted before its wrapper is recovered without refitting. Results remain `incoming_unreviewed`; filesystem readback is not a remote sharing-permission audit.

In [ ]:
if last_checkpoint is not None:
    print(json.dumps(last_checkpoint,indent=2))
else:
    print('No checkpoint exported. Enable EXPORT_IMMUTABLE_CHECKPOINT before training to preserve completed chunks.')


## 7 · Compare models, features and cost scenarios

Only configurations with every planned fold/seed task ready enter the validation ranking. The table retains partial and blocked evidence. The ranking is descriptive validation selection, not untouched-test evidence or permission to publish a model.

In [ ]:
if RUN_ONE_TRAINING_CHUNK:
    report = uv_module('src.research.colab_experiment_batch','report',
                       '--batch-spec',batch_file,'--data-root',stage_root,
                       '--output-root',output_root)
    # Figures use the frozen project environment, not the notebook kernel's packages.
    report_file = scratch/'validation-comparison.json'
    report_file.write_text(json.dumps(report),encoding='utf-8')
    figure = scratch/'validation-comparison.png'
    plot_script = """import json,sys,matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as p
r=json.load(open(sys.argv[1],encoding='utf-8'));rows=r['configurations']
complete=[x for x in rows if x['complete']]
shown=complete if complete else rows
values=[x['median_validation_net_excess']*100 for x in shown] if complete else [x['ready_tasks'] for x in shown]
fig,ax=p.subplots(figsize=(11,max(4,len(shown)*.29)))
ax.barh([x['configuration_id'] for x in shown],values,color='#3366aa');ax.invert_yaxis();ax.axvline(0,color='gray',linewidth=.6)
ax.set_xlabel('Median validation net excess vs QQQ (percentage points)' if complete else 'Completed fold-seed tasks (progress only)')
ax.set_title('Validation evidence only; holdout not evaluated' if complete else 'Incomplete configurations are excluded from ranking')
fig.tight_layout();fig.savefig(sys.argv[2],dpi=140);p.close(fig)
"""
    subprocess.run(['uv','run','--frozen','python','-c',plot_script,str(report_file),str(figure)],cwd=checkout,check=True)
    from IPython.display import display, Image, HTML
    from html import escape
    columns=['configuration_id','model_family','feature_set','phase','ready_tasks','planned_tasks','complete','median_validation_net_excess','worst_validation_drawdown']
    cells=lambda row: ''.join('<td>'+escape(str(row.get(k)))+'</td>' for k in columns)
    display(HTML('<table><thead><tr>'+''.join('<th>'+k+'</th>' for k in columns)+'</tr></thead><tbody>'+''.join('<tr>'+cells(row)+'</tr>' for row in report['configurations'])+'</tbody></table>'))
    display(Image(filename=str(figure)))
    print('Selection criterion:',report['selection_metric'])
    print('Reserved final holdout:',report['final_holdout'],'evaluated:',report['holdout_evaluated'])
    print('Incomplete configurations never enter validation_ranking. Multiple comparisons and curated-universe/PIT limits remain.')
else:
    print('No comparison yet: training opt-in is OFF.')


## Acceptance notes

Do not claim Colab success until real model training, completed/blocked trial receipts, identical local baseline and actual resource observations exist. After the batch finishes, use the existing local Agent study review to inspect every run and cost assumption. Explicitly reserve a **separate, untouched final test** after hyperparameter selection. This notebook cannot certify licensing, point-in-time availability, survivorship-free universe selection or trade readiness.

The runtime is pinned to uv 0.9.10 and Python 3.12.12; code uses a sparse checkout and execution uses the frozen uv environment. Package versions, measured trial seconds and actual training row/session counts are retained. Hardware/performance equivalence still requires an owner-run Colab receipt. [Official Colab FAQ](https://research.google.com/colaboratory/faq.html).

This preparation implements bounded seeded random search, not Bayesian/Optuna optimization. The 292-session full-US87 history limits information density; more trials do not create independent observations. CPU/GPU comparison, deeper models, CN factor applicability and final holdout execution require their own frozen follow-up evidence. No heavy sweep is executed locally by preparing this notebook.